In [2]:
import numpy as np
import os
from helpers import load_csv_data

DATA_PATH = "data"

# Variables to delete by NAME (present in CSV headers)
USELESS_VARS = [
     "FMONTH", "IDATE", "IMONTH", "IDAY", "IYEAR",
    "DISPCODE", "SEQNO", "_PSU", "CTELENUM"
]

# Hyper-params
MISSING_ROW_THRESHOLD = 0.60     # drop rows with >60% missing

# ---------------- helpers ----------------
def read_header(csv_path):
    """Read first line as header list."""
    with open(csv_path, "r", encoding="utf-8") as f:
        header = f.readline().strip().split(",")
    return [h.strip().strip('"') for h in header]

def header_after_loader(header):
    """
    helpers.load_csv_data() removes the FIRST column (IDs) from x_* arrays.
    So we drop header[0] to align names with returned matrices.
    """
    return header[1:]

def indices_for_names(header, names):
    """Return indices in header matching any of names (ignore absent)."""
    name_to_idx = {n: i for i, n in enumerate(header)}
    return sorted([name_to_idx[n] for n in names if n in name_to_idx])

def drop_columns_by_idx(X, idxs):
    if not idxs:
        return X
    keep = [i for i in range(X.shape[1]) if i not in set(idxs)]
    return X[:, keep]


    
def corr_pairwise_nan(X: np.ndarray) -> np.ndarray:
    """
    Matrice de corrélation de Pearson (pairwise) robuste aux NaN.
    - Si moins de 2 paires valides: corr = 0.0
    - Si l'écart-type d'une variable est 0: corr = 0.0
    """
    n, d = X.shape
    C = np.eye(d, dtype=float)

    for i in range(d):
        xi = X[:, i]
        for j in range(i + 1, d):
            xj = X[:, j]
            m = np.isfinite(xi) & np.isfinite(xj)
            if m.sum() < 2:
                r = 0.0
            else:
                a = xi[m] - np.mean(xi[m])
                b = xj[m] - np.mean(xj[m])
                sa = a.std(ddof=0)
                sb = b.std(ddof=0)
                if sa == 0.0 or sb == 0.0:
                    r = 0.0
                else:
                    r = float(np.mean(a * b) / (sa * sb))
            C[i, j] = C[j, i] = r
    return C


def drop_high_corr(X: np.ndarray, thr: float = 0.95):
    """
    Supprime une seule feature dans chaque paire très corrélée (|corr| >= thr).
    Stratégie gloutonne: on garde l'index le plus petit (colonne 'i'), on drop 'j'.

    Returns:
        X_reduced: np.ndarray   -> X[:, keep_idx]
        keep_idx: np.ndarray    -> indices conservés (triés)
        drop_idx: np.ndarray    -> indices supprimés (triés)
        C: np.ndarray           -> matrice de corrélation (utile pour debug)
    """
    d = X.shape[1]
    if d <= 1:
        return X, np.arange(d), np.array([], dtype=int), [], np.eye(d)

    C = corr_pairwise_nan(X)

    drop_mask = np.zeros(d, dtype=bool)
    corr_pairs = []  # store correlated pairs for logging

    for i in range(d):
        if drop_mask[i]:
            continue
        for j in range(i + 1, d):
            if drop_mask[j]:
                continue
            corr_val = C[i, j]
            if abs(corr_val) >= thr:
                drop_mask[j] = True
                corr_pairs.append((i, j, corr_val, j))  # (kept, dropped, correlation, dropped_index)

    keep_idx = np.where(~drop_mask)[0]
    drop_idx = np.where(drop_mask)[0]
    X_reduced = X[:, keep_idx]

    return X_reduced, keep_idx, drop_idx, corr_pairs, C

def zscore_fit(X):
    mu = np.nanmean(X, axis=0)
    sd = np.nanstd(X, axis=0)
    sd[sd == 0] = 1.0
    return mu, sd

def zscore_apply(X, mu, sd):
    return (X - mu) / sd

def unique_count_nonan(col):
    return np.unique(col[~np.isnan(col)]).size

def choose_cat_wide_cols(X):
    """Heuristic: one-hot if MIN_ONEHOT <= #unique <= MAX_ONEHOT."""
    d = X.shape[1]
    cat_idx = []
    cont_idx = []
    for j in range(d):
        k = unique_count_nonan(X[:, j])
        if 0 <= k <= 15:
            cat_idx.append(j)
        else:
            cont_idx.append(j)
    return cont_idx, cat_idx

def fit_one_hot_specs(X, cat_idx):
    """Learn category lists on TRAIN for each categorical col."""
    specs = {}
    for j in cat_idx:
        vals = np.unique(X[:, j][~np.isnan(X[:, j])])
        specs[j] = vals  # fixed category order
    return specs

def transform_one_hot(X, specs, d_total):
    """
    Expand categorical columns per learned specs, keep other columns as-is.
    Returns expanded matrix and a mapping for reproducibility.
    """
    parts = []
    for j in range(d_total):
        col = X[:, j]
        if j in specs:
            cats = specs[j]
            O = np.zeros((X.shape[0], len(cats)))
            # fill (NaNs -> all zeros)
            for i, v in enumerate(col):
                if not np.isnan(v):
                    # find index (ignore unseen cats -> all zeros)
                    pos = np.searchsorted(cats, v)
                    if pos < len(cats) and cats[pos] == v:
                        O[i, pos] = 1.0
            O = O[:, :-1]
            parts.append(O)
        else:
            parts.append(col.reshape(-1, 1))
    return np.hstack(parts)

def fill_nans_with_median(train_data, test_data, cont_idx):
    """
    Fill NaN values in continuous columns with the median value
    learned from the training set.

    Args:
        train_data (np.ndarray): training data matrix
        test_data (np.ndarray): test data matrix
        cont_idx (list[int]): list of continuous column indices

    Returns:
        train_filled (np.ndarray): training data with NaNs replaced
        test_filled (np.ndarray): test data with NaNs replaced
    """
    train_filled = train_data.copy()
    test_filled = test_data.copy()

    for j in cont_idx:
        col = train_data[:, j]
        median = np.nanmedian(col)
        
        # Replace NaN with median
        train_filled[np.isnan(train_data[:, j]), j] = median
        test_filled[np.isnan(test_data[:, j]), j] = median

    return train_filled, test_filled


def fill_nans_with_mode(train_data, test_data, cat_idx):
    """
    Fill NaN values in categorical columns with the most frequent (mode) value
    learned from the training set.
    
    Args:
        train_data (np.ndarray): training data matrix
        test_data (np.ndarray): test data matrix
        cat_idx (list[int]): list of categorical column indices
    
    Returns:
        train_filled (np.ndarray): training data with NaNs replaced
        test_filled (np.ndarray): test data with NaNs replaced
    """
    train_filled = train_data.copy()
    test_filled = test_data.copy()

    for j in cat_idx:
        col = train_data[:, j]
        valid = col[~np.isnan(col)]
        
        values, counts = np.unique(valid, return_counts=True)
        mode = values[np.argmax(counts)]

        # Replace NaNs with the mode (same mode for train & test)
        train_filled[np.isnan(train_data[:, j]), j] = mode
        test_filled[np.isnan(test_data[:, j]), j] = mode

    return train_filled, test_filled

In [3]:
x_train, x_test, y_train, train_ids, test_ids = load_csv_data(DATA_PATH)
print(f"[START] x_train: {x_train.shape}")

[START] x_train: (328135, 321)


In [13]:
    # Read headers (align with arrays by removing first column = ID)
h_train = header_after_loader(read_header(os.path.join(DATA_PATH, "x_train.csv")))
h_test  = header_after_loader(read_header(os.path.join(DATA_PATH, "x_test.csv")))
assert len(h_train) == x_train.shape[1], "Header/train mismatch after ID removal"
assert len(h_test)  == x_test.shape[1],  "Header/test mismatch after ID removal"

    # 1) Drop useless vars by NAME on both train and test
drop_idx_train = indices_for_names(h_train, USELESS_VARS)
drop_idx_test  = indices_for_names(h_test,  USELESS_VARS)
    # sanity: ensure same set by names; if not identical, drop by names independently
x_train_1 = drop_columns_by_idx(x_train, drop_idx_train)
x_test_1  = drop_columns_by_idx(x_test,  drop_idx_test)
    # update headers after drop
h_train = [c for i, c in enumerate(h_train) if i not in set(drop_idx_train)]
h_test  = [c for i, c in enumerate(h_test)  if i not in set(drop_idx_test)]
print(f"[AFTER drop useless vars] x_train: {x_train_1.shape}")

[AFTER drop useless vars] x_train: (328135, 312)


In [14]:
missing_values_threshold = 0.6
columns_to_drop = []
for i in range(x_train_1.shape[1]):
    missing_values = np.isnan(x_train_1[:, i]).sum() / x_train_1.shape[0]
    if missing_values > missing_values_threshold:
            columns_to_drop.append(i)
x_train_cleaned = np.delete(x_train_1, columns_to_drop, axis=1)
x_test_cleaned = np.delete(x_test_1, columns_to_drop, axis=1)
    

# Remove the same columns from the headers
x_train_header_cleaned = [col for i, col in enumerate(h_train) if i not in columns_to_drop]
x_test_header_cleaned = [col for i, col in enumerate(h_test) if i not in columns_to_drop]
    
    
print(f"[AFTER drop rows with >{MISSING_ROW_THRESHOLD*100}% missing] x_train: {x_train_cleaned.shape}")

[AFTER drop rows with >60.0% missing] x_train: (328135, 183)


In [15]:
# Threshold: if 95% or more of values are identical → drop
CONSTANT_RATIO_THRESHOLD = 0.95

columns_to_drop = []

for i in range(x_train_cleaned.shape[1]):
    col = x_train_cleaned[:, i]
    
    # Ignore NaNs for the check
    valid = col[~np.isnan(col)]

    # Compute frequency of the most common value
    unique_vals, counts = np.unique(valid, return_counts=True)
    most_common_ratio = np.max(counts) / valid.size

    # Drop if column is constant or quasi-constant
    if most_common_ratio >= CONSTANT_RATIO_THRESHOLD:
        columns_to_drop.append(i)

# Apply same drop to both train/test
x_train_cleaned = np.delete(x_train_cleaned, columns_to_drop, axis=1)
x_test_cleaned = np.delete(x_test_cleaned, columns_to_drop, axis=1)

# Clean headers as well
x_train_header_cleaned = [col for i, col in enumerate(x_train_header_cleaned) if i not in columns_to_drop]
x_test_header_cleaned = [col for i, col in enumerate(x_test_header_cleaned) if i not in columns_to_drop]

print(f"[AFTER dropping quasi-constant columns ≥{CONSTANT_RATIO_THRESHOLD*100:.0f}% identical] "
      f"dropped: {len(columns_to_drop)} columns | x_train: {x_train_cleaned.shape}")


[AFTER dropping quasi-constant columns ≥95% identical] dropped: 12 columns | x_train: (328135, 171)


In [7]:
#3) Remove highly correlated features (based on TRAIN), mirror on TEST by position
x_train_cleaned, keep_idx, drop_idx, corr_pairs, C = drop_high_corr(x_train_cleaned, thr=0.95)

print("Kept indices:", keep_idx)
print("Dropped indices:", drop_idx)
print("\nHighly correlated pairs (kept, dropped, corr, dropped_index):")
for kept, dropped, corr_val, dropped_idx in corr_pairs:
    kept_name = x_train_header_cleaned[kept]
    dropped_name = x_train_header_cleaned[dropped]
    print(f"{kept_name} ↔ {dropped_name} | corr = {corr_val:.3f} | dropped: {dropped_name}")

x_test_cleaned = x_test_cleaned[:, keep_idx]
x_train_header_cleaned = [h_train[i] for i in keep_idx]
x_test_header_cleaned  = [h_test[i]  for i in keep_idx]

Kept indices: [  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  77  80  81  82  83  84  85  86  87  88  89  90  91  93  95
  96  97  99 100 104 106 107 108 109 110 112 114 115 116 117 120 121 122
 123 124 125 126 127 129 130 131 132 137 138 139 141 142 145 146 147 148
 149 150 151 152 153 154 155 156 157 158 162 163 164 165 166 167 168]
Dropped indices: [ 29  76  78  79  92  94  98 101 102 103 105 111 113 118 119 128 133 134
 135 136 140 143 144 159 160 161 169 170]

Highly correlated pairs (kept, dropped, corr, dropped_index):
_STATE ↔ _STSTR | corr = 1.000 | dropped: _STSTR
NUMADULT ↔ _RAWRAKE | corr = 0.965 | dropped: _RAWRAKE
CADULT ↔ SEX | corr = 0.985 | dropped: SEX
HAVARTH3 ↔ _DRDXAR1 | corr = 1.000 | dropped: _DRDXAR1
EDUCA ↔ 

In [ ]:
# 4) Scale continuous & 5) One-hot “wide” categoricals (fit on TRAIN only)
cont_idx, cat_idx = choose_cat_wide_cols(x_train_cleaned)
print(f"Chosen {len(cont_idx)} continuous and {len(cat_idx)} categorical (one-hot) columns.")
# Manually correct feature type misclassifications
manual_cats = ["_STATE", "EXRACT11"]

# Find their indices in the header list
manual_cat_idx = [i for i, name in enumerate(x_train_header_cleaned) if name in manual_cats]

# Remove them from continuous indices (if present)
cont_idx = [i for i in cont_idx if i not in manual_cat_idx]

# Add them to categorical indices (avoid duplicates)
cat_idx = sorted(list(set(cat_idx + manual_cat_idx)))

print(f"Updated feature type assignment:")
print(f"  Continuous: {len(cont_idx)}")
print(f"  Categorical: {len(cat_idx)} (added {manual_cats})")

x_train_cleaned, x_test_cleaned = fill_nans_with_median(x_train_cleaned, x_test_cleaned,cont_idx)
x_train_filled, x_test_filled = fill_nans_with_mode(x_train, x_test, cat_idx)

# 4a) scale continuous (fit on train, apply to test with same indices)
if cont_idx:
    mu, sd = zscore_fit(x_train_cleaned[:, cont_idx])
    x_train_cleaned[:, cont_idx] = zscore_apply(x_train_cleaned[:, cont_idx], mu, sd)
    x_test_cleaned[:,  cont_idx] = zscore_apply(x_test_cleaned[:,  cont_idx],  mu, sd)



Chosen 64 continuous and 107 categorical (one-hot) columns.
Updated feature type assignment:
  Continuous: 62
  Categorical: 109 (added ['_STATE', 'EXRACT11'])


In [9]:
# 5a) fit one-hot specs on TRAIN and transform both sets
specs = fit_one_hot_specs(x_train_cleaned, cat_idx)
x_train_cleaned = transform_one_hot(x_train_cleaned, specs, d_total=len(x_train_header_cleaned))
x_test_cleaned  = transform_one_hot(x_test_cleaned,  specs, d_total=len(x_test_header_cleaned))
print(f"[AFTER scaling + one-hot] x_train: {x_train_cleaned.shape}")


[AFTER scaling + one-hot] x_train: (328135, 537)


In [ ]:
import numpy as np

def oversample_minority(X, y, target_ratio, random_state=42):
    """
    Oversample the minority class to reach a given ratio of the majority class size.

    Args:
        X (np.ndarray): Training features of shape (N, D)
        y (np.ndarray): Labels (-1, 1)
        target_ratio (float): Desired minority/majority ratio (e.g., 0.3 → 30%)
        random_state (int): Seed for reproducibility

    Returns:
        X_res (np.ndarray): Oversampled feature matrix
        y_res (np.ndarray): Corresponding oversampled labels
    """
    rng = np.random.default_rng(random_state)

    # Identify minority / majority classes
    labels, counts = np.unique(y, return_counts=True)
    minority_label = labels[np.argmin(counts)]
    majority_label = labels[np.argmax(counts)]

    idx_min = np.where(y == minority_label)[0]
    idx_maj = np.where(y == majority_label)[0]

    n_min = len(idx_min)
    n_maj = len(idx_maj)
    target_min = int(target_ratio * n_maj)

    # Choose random samples (with replacement)
    n_to_add = target_min - n_min
    new_idx = rng.choice(idx_min, size=n_to_add, replace=True)

    # Concatenate new samples after the original data (no shuffle)
    X_res = np.vstack([X, X[new_idx]])
    y_res = np.concatenate([y, y[new_idx]])

    return X_res, y_res


In [26]:
# After preprocessing, before model training:
x_train_cleaned_up, y_train_up = oversample_minority(x_train_cleaned, y_train, target_ratio=0.3)

print(f"Before: {np.sum(y_train==1)} positive, {np.sum(y_train==-1)} negative")
print(f"After:  {np.sum(y_train_up==1)} positive, {np.sum(y_train_up==-1)} negative")


Before: 28975 positive, 299160 negative
After:  89748 positive, 299160 negative


In [ ]:
# Save cleaned data with headers
import csv


with open('data/x_train_cleaned.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(x_train_header_cleaned)
    writer.writerows(x_train_cleaned_up)

with open('data/x_test_cleaned.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(x_test_header_cleaned)
    writer.writerows(x_test_cleaned)